In [7]:
import importlib
import SMILEStoSAFEconverter
importlib.reload(SMILEStoSAFEconverter)
from SMILEStoSAFEconverter import load_uspto_splits, smiles_to_safe, safe_to_smiles, validate_dataset, validate_dataset_parallel, _safe_encode_one
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

In [3]:
df = load_uspto_splits(
    train_path="Data/train.csv",
    val_path="Data/val.csv",
    test_path="Data/test.csv",
)
df["rxn_smiles"] = df["precursors"] + ">>" + df["products"]


print(df.columns.tolist())
print(df.head(3))

Loaded 409035 train / 30000 val / 40000 test = 479035 total rows
['precursors', 'products', 'split', 'rxn_smiles']
                                          precursors  \
0   C1CCOC1.CC(C)C[Mg+].CON(C)C(=O)c1ccc(O)nc1.[Cl-]   
1              CN.O.O=C(O)c1ccc(Cl)c([N+](=O)[O-])c1   
2  CCn1cc(C(=O)O)c(=O)c2cc(F)c(-c3ccc(N)cc3)cc21....   

                                           products  split  \
0                            CC(C)CC(=O)c1ccc(O)nc1  train   
1                    CNc1ccc(C(=O)O)cc1[N+](=O)[O-]  train   
2  CCn1cc(C(=O)O)c(=O)c2cc(F)c(-c3ccc(NC=O)cc3)cc21  train   

                                          rxn_smiles  
0  C1CCOC1.CC(C)C[Mg+].CON(C)C(=O)c1ccc(O)nc1.[Cl...  
1  CN.O.O=C(O)c1ccc(Cl)c([N+](=O)[O-])c1>>CNc1ccc...  
2  CCn1cc(C(=O)O)c(=O)c2cc(F)c(-c3ccc(N)cc3)cc21....  


In [8]:
results = validate_dataset_parallel(df, smiles_column="rxn_smiles")
print("Validation done")

N_WORKERS = max(1, cpu_count() - 1)

for split in ["train", "val", "test"]:
    subset = df[df["split"] == split].copy()
    with Pool(N_WORKERS) as pool:
        subset["safe"] = list(tqdm(
            pool.imap(_safe_encode_one, subset["rxn_smiles"], chunksize=64),
            total=len(subset),
            desc=f"Converting {split}"
        ))
    
    failed_mask = subset["safe"].isna()
    failed = failed_mask.sum()
    print(f"\n{split}: {failed} failed conversions out of {len(subset)}")
    
    if failed > 0:
        failed_df = subset[failed_mask][["rxn_smiles"]].copy()
        print(f"\nFailed reactions in {split}:")
        print(failed_df.to_string())
        failed_df.to_csv(f"Data/{split}_failed.csv", index=False)

    subset = subset.dropna(subset=["safe"])
    print(f"{split}: kept {len(subset)} / {len(subset) + failed} rows")
    subset.to_csv(f"Data/{split}_safe.csv", index=False)
    print(f"Saved Data/{split}_safe.csv")




Validating (21 workers): 100%|██████████| 479035/479035 [13:41<00:00, 583.38it/s] 



Total rows:       479035
Round-trip OK:    478998 (99.99%)
Encode failures:  37
Decode failures:  0
Mismatches:       0
Validation done


Converting train: 100%|██████████| 409035/409035 [06:28<00:00, 1052.25it/s]



train: 29 failed conversions out of 409035

Failed reactions in train:
                                                                                                                                                                                             rxn_smiles
8578                                                  CCNc1ccncc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccncc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
14987                           CCNc1ccc(NC(=O)COC)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccc(NC(=O)COC)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
21928                 CCNc1ccc(NC(=O)C(F)(F)F)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccc(NC(=O)C(F)(F)F)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
26962                                                                                   O=C(Cl)C(=O)Cl.O=C(O)C1CCN1Cc1ccccc1.[O-

Converting val: 100%|██████████| 30000/30000 [00:34<00:00, 875.12it/s] 



val: 2 failed conversions out of 30000

Failed reactions in val:
                                                                                                                                                                                 rxn_smiles
418971  CCNc1ccc(NC(=O)CN2CCOCC2)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccc(NC(=O)CN2CCOCC2)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
426870                                              Br.CC#N.CC(=O)O.CCOCC.COc1cc(C=O)c(C(=O)O)cc1OC.c1ccc(P(c2ccccc2)c2ccccc2)cc1>>COc1cc2c(cc1OC)C([PH2](c1ccccc1)(c1ccccc1)c1ccccc1)OC2=O
val: kept 29998 / 30000 rows
Saved Data/val_safe.csv


Converting test: 100%|██████████| 40000/40000 [00:44<00:00, 890.54it/s] 



test: 6 failed conversions out of 40000

Failed reactions in test:
                                                                                                                                                                               rxn_smiles
439723          CCNC(=O)c1ccc(NCC)c([N+](=O)[O-])c1.CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNC(=O)c1ccc(NCC)c(N=C2SC(=C3Sc4ccccc4N3C)C(=O)N2Cc2ccccc2)c1
440586                  CCOc1ccc(C(C)=O)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCOc1ccc(C(C)=O)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
441307                              COc1ccc(-c2nc([CH2+](CBr)C(F)(F)CF)[nH]c2-c2ccc(OC)cc2)cc1.FC(Br)C(F)(F)Sc1nc(-c2ccccc2)c(-c2ccccc2)[nH]1>>FCC(F)(F)Sc1nc(-c2ccccc2)c(-c2ccccc2)[nH]1
456560  CCNc1ccc(C(=O)OC(C)(C)C)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccc(C(=O)OC(C)(C)C)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
46

In [5]:
encode_failures = results["encode_fail"]
print(f"Total encode failures: {len(encode_failures)}")
for i, rxn, error in encode_failures[:20]:
    print(f"\nRow {i}: {rxn}")
    print(f"Error: {error}")

# Save to CSV

failures_df = pd.DataFrame(encode_failures, columns=["row_index", "rxn_smiles", "error"])
failures_df.to_csv("Data/encode_failures.csv", index=False)
print(f"Saved {len(failures_df)} failures to Data/encode_failures.csv")


Total encode failures: 37

Row 8578: CCNc1ccncc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccncc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
Error: SAFEEncodeError('Failed to encode CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1 with brics')

Row 14987: CCNc1ccc(NC(=O)COC)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccc(NC(=O)COC)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
Error: SAFEEncodeError('Failed to encode CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1 with brics')

Row 21928: CCNc1ccc(NC(=O)C(F)(F)F)cc1[N+](=O)[O-].CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1.Cc1ccc(S(=O)(=O)[O-])cc1>>CCNc1ccc(NC(=O)C(F)(F)F)cc1N=C1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1
Error: SAFEEncodeError('Failed to encode CS[CH2+]1SC(=C2Sc3ccccc3N2C)C(=O)N1Cc1ccccc1 with brics')

Row 26962: O=C(Cl)C(=O)Cl.O=C(O)C1CCN1Cc1ccccc1.[O-][Cl+3]([O-])([O-])O>>[O-][Cl+3]([O-])([O-])[O-].c1ccc(CN2CC[CH3+]2)cc1
Error: SAFEEncodeError('Fa

In [ ]:
# Debugging and error analysis
encode_failures = results["encode_fail"]
# Print first 5 errors
for i, rxn, error in encode_failures[:5]:
    print(f"\nRow {i}:")
    print(f"RXN: {rxn}")
    print(f"Error: {error}")



